# 04 — Proxy-blocked 5-fold cross-validation

**CAVEAT (Opus ESC-20260908-bb1eb1):** This is PROXY-blocked CV, not subject-blocked.
It inherits the proxy grouping defect exposed by GATE 2 (74.2% residual train-to-test
leakage dose on ground-truth subject IDs). Every result from it carries this caveat.

Blocked 5-fold CV tests **every image exactly once**: the Benign estimate rests on
all 120 slices instead of 12, eliminating single-draw test variance.

Budget ~3 GPU-h for two models (effnet_b0, ccal).

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import numpy as np
recs = splits.build_registry(ROOT)
OUT = f"{RESULTS}/04_cv"; os.makedirs(OUT, exist_ok=True)
MODELS = ["effnet_b0", "ccal"]
N_FOLDS, SEED = 5, 42

for model in MODELS:
    for fold, rs in splits.blocked_kfold(recs, n_folds=N_FOLDS, seed=SEED):
        for r in rs:
            r["strategy"] = f"proxy_blocked_kfold_{N_FOLDS}"
        name = f"{model}__proxy_blocked_kfold_{N_FOLDS}__seed{SEED}__fold{fold}"
        if os.path.exists(f"{OUT}/{name}_result.json"):
            print("skip", name); continue
        engine.run_experiment(ROOT, model, strategy=None, seed=SEED, out_dir=OUT,
                              records=[dict(r) for r in rs], tag=f"fold{fold}")


In [ ]:
# Pool predictions across folds -> one evaluation over all 1097 images.
for model in MODELS:
    yt, yp, bids = [], [], []
    for fold in range(N_FOLDS):
        f = [x for x in os.listdir(OUT)
             if x.startswith(model) and f"fold{fold}" in x and x.endswith("_preds.npz")]
        if not f: continue
        d = metrics.load_predictions(f"{OUT}/{f[0]}")
        yt.append(d["y_true"]); yp.append(d["y_pred"]); bids.append(d["block_ids"])
    if not yt: continue
    yt, yp, bids = np.concatenate(yt), np.concatenate(yp), np.concatenate(bids)
    print(f"
### {model} (Proxy-Blocked CV) pooled n={len(yt)}")
    s = metrics.summarise_run(yt, yp, block_ids=bids)
    print(f"  accuracy  {s['accuracy']:.2f}  CI [{s['accuracy_ci']['lo']:.2f}, "
          f"{s['accuracy_ci']['hi']:.2f}]  (cluster bootstrap over blocks)")
    print(f"  BenignF1  {s['benign_f1']:.2f}  CI [{s['benign_f1_ci']['lo']:.2f}, "
          f"{s['benign_f1_ci']['hi']:.2f}]")
    for c, v in s["per_class"].items():
        print(f"    {c:<10} P {v['precision']:6.2f}  R {v['recall']:6.2f}  "
              f"F1 {v['f1']:6.2f}  n={v['support']}")
    with open(f"{RESULTS}/04_proxy_blocked_pooled_{model}.json", "w") as fh:
        json.dump(s, fh, indent=2)
